In [1]:
import sys
from pathlib import Path

root = Path().resolve()
sys.path.insert(0, str(root / "src"))


In [2]:
from FLRW_Net.networks.test import GeneralizedModel
import tensorflow as tf
from FLRW_Net.layers.strut_activation import StrutActivation
from FLRW_Net.layers.spatial_edge_activation import SpatialEdgeActivation

In [ ]:

tf.keras.backend.set_floatx('float64')

slice_specs_3 = [
    (0, 1),   # l1
    (0, 3),   # first 3 inputs
    (0, 5),   # first 5 inputs
    (2, 5),   # inputs[2:5]
    (2, 7),   # inputs[2:7]
    (4, 7),   # inputs[4:7]
    (6, 7),   # l4
]

slice_specs_2 = [
    (0, 1),   # l1
    (0, 3),   # first 3 inputs
    (0, 5),   # full input, adjust if necessary
    (2, 5),   # inputs[2:]
    (4, 5),   # l3
]

inputs_3 = tf.constant([[1, 0.5, 2, 0.5, 3, 0.6, 4]], dtype=tf.float64)
inputs_2 = tf.constant([[1, 0.5, 3.5, 0.5, 4]], dtype=tf.float64)

In [52]:
make_slice_specs(3)

[(0, 1), (0, 3), (2, 5), (4, 7), (0, 5), (2, 7), (6, 7)]

In [49]:
def make_slice_specs(n: int) -> list[tuple[int, int]]:
    length = 2*n-1
    specs = []

    # First element
    specs.append((0, 1))

    # Sliding windows of size 3
    specs.extend((center-1, center+2) for center in range(1, length+1, 2))

    if length >= 2:  # noqa: PLR2004
        # Sliding windows of size 5
        specs.extend((center-2, center+3) for center in range(2, length, 2))

    # Last element
    specs.append((length+1, length+2))

    return specs


In [ ]:
model_3 = GeneralizedModel(slice_specs_3)
output_3 = model_3(inputs_3)

model_2 = GeneralizedModel(slice_specs_2)
output_2 = model_2(inputs_2)

In [ ]:
print(output_3)
print(output_2)

In [ ]:
indices = tf.range(1, 2*3+1, 2)

for idx in indices.numpy():  # gather slices
    # Get a slice of 3 features centered at idx
    start = idx - 1
    end = idx + 2
    slice = inputs_3[:, start:end]
    print(slice)

In [ ]:
tf.keras.backend.set_floatx('float64')
inputs = tf.constant([[1, 0.5, 3.5, 0.5, 4]], dtype=tf.float64)

strut_activation_layer = StrutActivation()

print(strut_activation_layer(inputs))

In [ ]:
import numpy as np

x = np.sqrt(0.6 +3/8)

print(f"{x:.15f}")

In [ ]:
indices = tf.range(1, 7, 2)

# Compute start and end indices for 3-feature slices
start_indices = indices - 1
end_indices = indices + 2

print(indices)
print(start_indices)
print(end_indices)

In [ ]:
indices = tf.range(2, 9-2, 2)

# Compute start and end indices for 3-feature slices
start_indices = indices - 2
end_indices = indices + 3

print(indices)
print(start_indices)
print(end_indices)

In [6]:
tf.keras.backend.set_floatx('float64')
spatial_edge_layer = SpatialEdgeActivation()
inputs_3 = tf.constant([[1, 0.5, 2, 0.5, 3, 0.6, 4]], dtype=tf.float64)
inputs_2 = tf.constant([[1, 0.5, 3.5, 0.5, 4]], dtype=tf.float64)

print(spatial_edge_layer(inputs_3))
print(spatial_edge_layer(inputs_2))

tf.Tensor(
[[1.         0.5        4.         0.5        4.54166667 0.6
  4.        ]], shape=(1, 7), dtype=float64)
tf.Tensor([[ 1.     0.5   15.625  0.5    4.   ]], shape=(1, 5), dtype=float64)


In [8]:
n: tf.Tensor = tf.shape(inputs_3)[1]
output: tf.Tensor = tf.identity(inputs_3)

interior_indices: tf.Tensor = tf.range(1, n-1)
odd_indices: tf.Tensor = interior_indices[interior_indices % 2 == 1]
even_indices: tf.Tensor = interior_indices[interior_indices % 2 == 0]

print(odd_indices)
print(even_indices)

tf.Tensor([1 3 5], shape=(3,), dtype=int32)
tf.Tensor([2 4], shape=(2,), dtype=int32)


In [ ]:
inputs_4 = tf.constant([[1, 2, 2.2, 4, 3.1, 6, 7]], dtype=tf.float64)
print(inputs_4)

n: tf.Tensor = tf.shape(inputs_3)[1]
output: tf.Tensor = tf.identity(inputs_3)

odd_indices: tf.Tensor = tf.range(1, n-1, 2)
even_indices: tf.Tensor = tf.range(2, n-1, 2)

scaled_as: tf.Tensor = tf.gather(inputs_3, odd_indices, axis=1)
updated_spatial_edges: tf.Tensor = tf.gather(inputs_4, even_indices, axis=1)

# Scatter odd values from input_1
odd_scatter_indices = tf.stack([tf.zeros_like(odd_indices), odd_indices], axis=1)
output = tf.tensor_scatter_nd_update(output, odd_scatter_indices, tf.reshape(scaled_as, [-1]))

# Scatter even values from input_2
even_scatter_indices = tf.stack([tf.zeros_like(even_indices), even_indices], axis=1)
output = tf.tensor_scatter_nd_update(output, even_scatter_indices, tf.reshape(updated_spatial_edges, [-1]))

tf.Tensor([[1.  2.  2.2 4.  3.1 6.  7. ]], shape=(1, 7), dtype=float64)
tf.Tensor([[2.2 3.1]], shape=(1, 2), dtype=float64)
tf.Tensor(
[[0 1]
 [0 3]
 [0 5]], shape=(3, 2), dtype=int32)
tf.Tensor([[1.  0.5 2.  0.5 3.  0.6 4. ]], shape=(1, 7), dtype=float64)
tf.Tensor(
[[0 2]
 [0 4]], shape=(2, 2), dtype=int32)
tf.Tensor([[1.  0.5 2.2 0.5 3.1 0.6 4. ]], shape=(1, 7), dtype=float64)


In [ ]:
import tensorflow as tf

InvalidArgumentError: {{function_node __wrapped__Pack_N_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Shapes of all inputs must match: values[0].shape = [1,3] != values[1].shape = [1,2] [Op:Pack] name: stack

In [7]:
a = tf.constant([[1, 3, 5, 4, 6, 2]])
b = tf.constant([[2, 9]])

combined = tf.concat([a, b], axis=1)
tf.reduce_mean(combined, axis=1)

<tf.Tensor: shape=(1,), dtype=int32, numpy=array([4])>